In [ ]:
#Import packages 

import numpy as np 
import geopandas as gpd 
import matplotlib.pyplot as plt 
from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors
import pandas as pd 
from shapely.geometry import shape 
import json 
from shapely import wkt 
from shapely.geometry import Point
from shapely.geometry import box
from math import cos, radians
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.cm import ScalarMappable
import seaborn as sns 
import matplotlib
from statsmodels.tsa.seasonal import seasonal_decompose

import glob
import os
import csv
import ast

from scipy.stats import chi2_contingency
from math import sqrt

### Functions - Global 

In [3]:
def assign_season(month):
    if month in [12, 1, 2, 3]:
        return 'Dry_Season'
    elif month in [4, 5, 6, 7]:
        return 'Major_Rainy_Season'
    elif month in [9, 10, 11]:
        return 'Minor_Rainy_Season'
    else:
        return 'Transition_Season'

### Reading Temp & Precip (6km buffer)

In [38]:
temp_all = pd.read_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/temp_all_years_6km_buffer.csv').drop(columns = ['Unnamed: 0'])

temp_all['time'] = pd.to_datetime(temp_all['time'])

In [39]:
precip_all = pd.read_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/precip_all_years_6km_buffer.csv').drop(columns = ['Unnamed: 0'])

precip_all['time'] = pd.to_datetime(precip_all['time'])

### Wind (6km buffer)

In [ ]:
hourly_wind_df = pd.read_csv('/home/kdonkor_umass_edu/Interpolation/ea_hourly_wind_avg_6km.csv').drop(columns=['Unnamed: 0'])

hourly_wind_df['Timestamp'] = pd.to_datetime(hourly_wind_df['Timestamp'])
hourly_wind_df = hourly_wind_df.rename(columns = {'Timestamp':'time'})
hourly_wind_df['Date'] = hourly_wind_df['time'].dt.date

### Lightning (aligned with 6km buffers)

In [ ]:
hourly_lightning_df = pd.read_csv('/home/kdonkor_umass_edu/Interpolation/ea_hourly_lightning_avg_6km_buffer_aligned.csv').drop(columns=['Unnamed: 0'])

hourly_lightning_df['Timestamp'] = pd.to_datetime(hourly_lightning_df['Timestamp'])
hourly_lightning_df = hourly_lightning_df.rename(columns = {'Timestamp':'time'})
hourly_lightning_df['Date'] = hourly_lightning_df['time'].dt.date

### Extreme Definitions 

### Temp 

In [47]:
temp_all['Date'] = temp_all['time'].dt.floor('D')

# Hot hour flag
temp_all['Hot_Hour'] = temp_all['Temp'] > 32

# Group by EA and Date, and count Hot_Hour sum
daily_hot_hours = (
    temp_all.groupby(['ea_code9ch', 'Date'], as_index=False)
            .agg({'Hot_Hour': 'sum'})
)

daily_hot_hours = daily_hot_hours.rename(columns={'Hot_Hour': 'Num_Hot_Hours'})

### Precip 

In [49]:
precip_all['Date'] = precip_all['time'].dt.floor('D')  

# Group by EA and Date, sum Precip
daily_precip = (
    precip_all.groupby(['ea_code9ch', 'Date'], as_index=False)
      .agg({'Precip': 'sum'})
)

### Wind 

In [51]:
# Windy hour flag
hourly_wind_df['Windy_Hour'] = hourly_wind_df['Wind Gusts (m/s)'] > 5.93

# Group by EA and Date, and count Windy_Hour sum
daily_windy_hours = (
    hourly_wind_df.groupby(['ea_code9ch', 'Date'], as_index=False)
            .agg({'Windy_Hour': 'sum'})
)

daily_windy_hours = daily_windy_hours.rename(columns={'Windy_Hour': 'Num_Windy_Hours'})

### Lightning 

In [53]:
daily_lightning_per_ea = hourly_lightning_df.groupby(['ea_code9ch', 'Date'])['Lightning Events'].sum().reset_index()

### Extreme Weather Percentiles 

In [ ]:
#### 90th percentile 

hot_hrs_90_thresh = daily_hot_hours['Num_Hot_Hours'].quantile(0.90)
temp_90_thresh = temp_all['Temp'].quantile(0.90)

precip_90_thresh = daily_precip['Precip'].quantile(0.90)

# this resulted to 0 for lightning
lightning_90_thresh = daily_lightning_per_ea['Lightning Events'].quantile(0.90)

windy_hrs_90_thresh = daily_windy_hours['Num_Windy_Hours'].quantile(0.90)
wind_90_thresh = hourly_wind_df['Wind Gusts (m/s)'].quantile(0.90)

# --- # 

#### 95th percentile 

hot_hrs_95_thresh = daily_hot_hours['Num_Hot_Hours'].quantile(0.95)
temp_95_thresh = temp_all['Temp'].quantile(0.95)

precip_95_thresh = daily_precip['Precip'].quantile(0.95)

lightning_95_thresh = daily_lightning_per_ea['Lightning Events'].quantile(0.95)

windy_hrs_95_thresh = daily_windy_hours['Num_Windy_Hours'].quantile(0.95)
wind_95_thresh = hourly_wind_df['Wind Gusts (m/s)'].quantile(0.95)

# --- # 

#### 99th percentile 

hot_hrs_99_thresh = daily_hot_hours['Num_Hot_Hours'].quantile(0.99)
temp_99_thresh = temp_all['Temp'].quantile(0.99)

precip_99_thresh = daily_precip['Precip'].quantile(0.99)

lightning_99_thresh = daily_lightning_per_ea['Lightning Events'].quantile(0.99)

windy_hrs_99_thresh = daily_windy_hours['Num_Windy_Hours'].quantile(0.99)
wind_99_thresh = hourly_wind_df['Wind Gusts (m/s)'].quantile(0.99)


#### Dictionary for percentiles 

In [60]:
# Percentile Threshold dictionaries 
temp_thresh_dict = {
    '90': temp_90_thresh,
    '95': temp_95_thresh,
    '99': temp_99_thresh
}

hot_hrs_thresh_dict = {
    '90': hot_hrs_90_thresh,
    '95': hot_hrs_95_thresh,
    '99': hot_hrs_99_thresh
}

precip_thresh_dict = {
    '90': precip_90_thresh,
    '95': precip_95_thresh,
    '99': precip_99_thresh
}

wind_thresh_dict = {
    '90': wind_90_thresh,
    '95': wind_95_thresh,
    '99': wind_99_thresh
}

windy_hrs_thresh_dict = {
    '90': windy_hrs_90_thresh,
    '95': windy_hrs_95_thresh,
    '99': windy_hrs_99_thresh
}

lightning_thresh_dict = {
    '90': 1,   # at least 1 lightning strike 
    '95': lightning_95_thresh,
    '99': lightning_99_thresh
}

## --- Global Co_occurrence Workflow (TPLW dataset) --- 

### EAs n Sites Mapping (focus is on EAs that intersect with the 6km weather buffer) 

In [ ]:
merged_eas_sites = pd.read_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/ea_site_list_6km_buffer.csv')
merged_eas_sites = merged_eas_sites[['ea_code9ch', 'Intersecting_Sites']]

# Convert the string representation of lists to actual lists
merged_eas_sites['Intersecting_Sites'] = merged_eas_sites['Intersecting_Sites'].apply(ast.literal_eval)

## Outage Workflow 

### read hourly PQR data  

In [ ]:
## 22 
pqr_hourly_22 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/2022/merged_outage_n_voltage_hourly_22_NEW.csv')
pqr_hourly_22['time'] = pd.to_datetime(pqr_hourly_22['time'])
pqr_hourly_22['time'] = pqr_hourly_22['time'].dt.tz_convert(None)

## 23 
pqr_hourly_23 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/2023/merged_outage_n_voltage_hourly_23_NEW.csv')
pqr_hourly_23['time'] = pd.to_datetime(pqr_hourly_23['time'])
pqr_hourly_23['time'] = pqr_hourly_23['time'].dt.tz_convert(None)

# concatenate 
pqr_hourly_all = pd.concat([pqr_hourly_22, pqr_hourly_23], ignore_index=True).drop(columns=['Unnamed: 0'])

In [85]:
site_start_dates = pqr_hourly_all.groupby('site_id')['time'].min()

# Filter for sites that start in Jan 2022
sites_with_full_data = site_start_dates[site_start_dates.dt.to_period('M') == '2022-01'].index

monthly_sums = (
    pqr_hourly_all
    .groupby(['site_id', pd.Grouper(key='time', freq='ME')])  # replace with actual name
    .sum()
    .reset_index()
)

# remove site '0' 
monthly_sums = monthly_sums[~(monthly_sums['site_id'] == 0)].reset_index(drop = True)

### Remove sites with less than TWO YEARS worth of data 

In [ ]:
### function to remove sites with less than 2 years data 

def get_incomplete_sites(df, expected_months=24):
    incomplete_sites = []
    for site in df['site_id'].unique():
        count = df[df['site_id'] == site].shape[0]
        if count < expected_months:
            print(f"Site ID {site}: {count} months (incomplete)")
            incomplete_sites.append(site)
    return incomplete_sites

In [ ]:
# Sites with less than 24 months 
incomplete_site_ids = get_incomplete_sites(monthly_sums)

pqr_hourly_all = pqr_hourly_all[ ~(pqr_hourly_all['site_id'].isin(incomplete_site_ids)) ]

### Functions 

In [ ]:
## Specify Outage Duration (to be classified as outage or not) 

def flag_outage_hours(df, threshold):
    
    df = df.copy()
    df['Outage_Flag'] = df['outage_mins'] >= threshold
    df['Outage_Dur'] = round( (threshold/60), 2)
    df = df[['time', 'site_id', 'Outage_Flag', 'Outage_Dur']]
    
    return df

In [ ]:
def prepare_hourly_df_TPLW_n_outage_data(ea_row, temp_df, precip_df, lightning_df, wind_df, pqr_df, outage_threshold):
    ea = ea_row['ea_code9ch']
    site_list = ea_row['Intersecting_Sites']

    all_merged = []

    for site_id in site_list:
        site_id = int(site_id)

        # Filter temperature
        temp_filt = (
            temp_df[temp_df['ea_code9ch'] == ea]
            .reset_index(drop=True)
            .drop(columns=['geometry'], errors='ignore')
        )
        if 'Hot_Hour' in temp_filt.columns:
            temp_filt = temp_filt.drop(columns=['Hot_Hour'])

        temp_filt['time'] = temp_filt['time'].astype('datetime64[ns]')

        # Filter precipitation
        precip_filt = (
            precip_df[precip_df['ea_code9ch'] == ea]
            .reset_index(drop=True)
            .drop(columns=['geometry'], errors='ignore')
        )
        precip_filt['time'] = precip_filt['time'].astype('datetime64[ns]')
        precip_filt = precip_filt[['time', 'Precip']]

        # Merge temp and precip
        merged = temp_filt.merge(precip_filt, on='time', how='outer')

        # Filter lightning and outer join
        lightning_filt = (
            lightning_df[lightning_df['ea_code9ch'] == ea]
            .reset_index(drop=True)
            .drop(columns=['geometry'], errors='ignore')
            [['time', 'Lightning Events']]
        )
        merged = merged.merge(lightning_filt, on='time', how='outer')
        merged['Lightning Events'] = merged['Lightning Events'].fillna(0)

        ## Merge with Wind (inner merge) 
        wind_filt = (
            wind_df[wind_df['ea_code9ch'] == ea]
            .reset_index(drop=True)
            .drop(columns=['geometry'], errors='ignore')
        )
        wind_filt = wind_filt[['time', 'Wind Gusts (m/s)']]  
        wind_filt['time'] = wind_filt['time'].astype('datetime64[ns]')

        # Merge temp and precip
        merged = merged.merge(wind_filt, on='time', how='inner')
     
        # Filter outage data and flag
        pqr_filt = (
            pqr_df[pqr_df['site_id'] == site_id]
            .reset_index(drop=True)[['time', 'site_id', 'outage_events', 'outage_mins']]
        )
        flagged_outages = flag_outage_hours(pqr_filt, threshold=outage_threshold)

        # Inner join outages
        merged = merged.merge(flagged_outages, on='time', how='inner')

        # Add identifiers
        merged['ea_code9ch'] = ea
        merged['site_id'] = site_id

        # Reorder columns
        merged = merged[['time', 'Date', 'ea_code9ch', 'site_id', 'Temp', 'Precip', 
                 'Lightning Events', 'Wind Gusts (m/s)', 
                 'Outage_Flag', 'Outage_Dur']]

        all_merged.append(merged)

    return pd.concat(all_merged, ignore_index=True)

In [ ]:
### fix double counting 

def create_daily_summary(df, temp_thresh=32, hot_hours_thresh=5, precip_thresh=15, lightning_thresh = 42, wind_thresh = 5.93, windy_hours_thresh = 6):
    df = df.copy()
    df['Date'] = df['time'].dt.date
    df['Hot_hour'] = df['Temp'] > temp_thresh
    df['Windy_Hour'] = df['Wind Gusts (m/s)'] > wind_thresh

    daily_summary = (
        df.groupby(['Date', 'ea_code9ch'], as_index=False)
          .agg({
              'Outage_Flag': lambda x: (x > 0).any(),   # EA outage if any site outage
              'Outage_Dur': 'mean',
              'Precip': 'sum',
              'Lightning Events':'sum', 
              'Hot_hour': 'sum', 
              'Windy_Hour':'sum'
          })
    )

    ## Classify Days as High Temperature or High Precipitation 
    daily_summary['Hot_Day'] = daily_summary['Hot_hour'] >= hot_hours_thresh
    daily_summary['Windy_Day'] = daily_summary['Windy_Hour'] >= windy_hours_thresh
    
    daily_summary['Rainy_Day'] = daily_summary['Precip'] >= precip_thresh
    daily_summary['Extreme_Lightning_Day'] = daily_summary['Lightning Events'] >= lightning_thresh

    # Add Season Column
    daily_summary['Month'] = pd.to_datetime(daily_summary['Date']).dt.month
    daily_summary['Season'] = daily_summary['Month'].apply(assign_season)

    return daily_summary

In [ ]:
def create_daily_summary_exclusive(
    df,
    temp_thresh,
    hot_hours_thresh,
    precip_thresh,
    lightning_thresh, 
    wind_thresh, 
    windy_hours_thresh
):
    df = df.copy()
    df['Date'] = df['time'].dt.date
    df['Hot_hour'] = df['Temp'] > temp_thresh
    df['Windy_Hour'] = df['Wind Gusts (m/s)'] > wind_thresh

    daily_summary = (
        df.groupby(['Date', 'ea_code9ch'], as_index=False)
          .agg({
              'Outage_Flag': lambda x: (x > 0).any(),   # EA outage if any site outage
              'Outage_Dur': 'mean',
              'Precip': 'sum',
              'Lightning Events':'sum', 
              'Hot_hour': 'sum', 
              'Windy_Hour':'sum'
          })
    )

    # Classify days as weather events
    daily_summary['Hot_Day'] = daily_summary['Hot_hour'] >= hot_hours_thresh
    daily_summary['Windy_Day'] = daily_summary['Windy_Hour'] >= windy_hours_thresh
    daily_summary['Rainy_Day'] = daily_summary['Precip'] >= precip_thresh   # <- renamed here
    daily_summary['Extreme_Lightning_Day'] = daily_summary['Lightning Events'] >= lightning_thresh

    # Single exclusive events
    daily_summary['Hot_Day_Only'] = (
        daily_summary['Hot_Day'] &
        ~daily_summary['Rainy_Day'] &
        ~daily_summary['Extreme_Lightning_Day'] & 
        ~daily_summary['Windy_Day']
    )
    
    daily_summary['Rainy_Day_Only'] = (
        daily_summary['Rainy_Day'] &
        ~daily_summary['Hot_Day'] &
        ~daily_summary['Extreme_Lightning_Day'] & 
        ~daily_summary['Windy_Day']
    )
    
    daily_summary['Extreme_Lightning_Day_Only'] = (
        daily_summary['Extreme_Lightning_Day'] &
        ~daily_summary['Hot_Day'] &
        ~daily_summary['Rainy_Day'] & 
        ~daily_summary['Windy_Day']
    )
    
    daily_summary['Windy_Day_Only'] = (
        daily_summary['Windy_Day'] &
        ~daily_summary['Hot_Day'] &
        ~daily_summary['Rainy_Day'] & 
        ~daily_summary['Extreme_Lightning_Day']
    )

    # Combination exclusives
    T = daily_summary['Hot_Day']
    P = daily_summary['Rainy_Day']   # <- P now maps to Rainy
    L = daily_summary['Extreme_Lightning_Day']
    W = daily_summary['Windy_Day']

    # Two-way combinations
    daily_summary['TL_Only'] = (T & L) & ~P & ~W
    daily_summary['TP_Only'] = (T & P) & ~L & ~W
    daily_summary['TW_Only'] = (T & W) & ~P & ~L
    daily_summary['PL_Only'] = (P & L) & ~T & ~W
    daily_summary['PW_Only'] = (P & W) & ~T & ~L
    daily_summary['WL_Only'] = (W & L) & ~T & ~P

    # Three-way combinations
    daily_summary['TPL_Only'] = (T & P & L) & ~W
    daily_summary['TPW_Only'] = (T & P & W) & ~L
    daily_summary['TWL_Only'] = (T & W & L) & ~P
    daily_summary['PWL_Only'] = (P & W & L) & ~T

    # Four-way combination
    daily_summary['TPWL_Only'] = (T & P & W & L)

    # Drop the original multi-event flags if you only want exclusives + only-flags
    drop_cols = ['Hot_Day', 'Rainy_Day', 'Extreme_Lightning_Day', 'Windy_Day']
    daily_summary = daily_summary.drop(columns=[col for col in drop_cols if col in daily_summary.columns])

    return daily_summary

In [ ]:
### new 

from itertools import combinations

def calculate_cooccurrence_prob_all_combinations(
    df, 
    print_top_n=10, 
    filter_top_n_only=True, 
    print_names=True
):
    df = df.copy()

    # Normalize season labels (merge Major/Minor rainy)
    df['Season'] = df['Season'].replace({
        'Major_Rainy_Season': 'Rainy_Season',
        'Minor_Rainy_Season': 'Rainy_Season'
    })

    df['Outage_Day'] = df['Outage_Flag'] > 0

    # Define hazards and their columns (added Wind W)
    hazard_cols = {
        'T': 'Hot_Day',
        'P': 'Rainy_Day',
        'W': 'Windy_Day', 
        'L': 'Extreme_Lightning_Day'
    }
    hazard_keys = list(hazard_cols.keys())

    # Generate all non-empty combinations of hazards
    all_combos = []
    for r in range(1, len(hazard_keys) + 1):
        all_combos.extend(combinations(hazard_keys, r))

    # Track combo counts
    combo_counts = {}

    # Create exclusive columns for each combination
    for combo in all_combos:
        combo_name = ''.join(combo)
        in_combo = np.logical_and.reduce([df[hazard_cols[h]] for h in combo])
        not_in_combo = np.logical_not(
            np.logical_or.reduce([df[hazard_cols[h]] for h in hazard_keys if h not in combo])
        )
        df[f'{combo_name}_Only_Day'] = in_combo & not_in_combo
        df[f'{combo_name}_Only_Outage'] = df[f'{combo_name}_Only_Day'] & df['Outage_Day']
        combo_counts[combo_name] = df[f'{combo_name}_Only_Day'].sum()

    # Identify top-N most frequent combos
    top_combos = sorted(combo_counts.items(), key=lambda x: x[1], reverse=True)[:print_top_n]
    top_combo_names = {name for name, _ in top_combos}

    def p_cond(n, d): 
        return n / d if d else np.nan

    results = []

    # Per-season results
    for season in df['Season'].unique():
        season_df = df[df['Season'] == season]
        tot_days = len(season_df)

        row = {
            'season': season,
            'total_days': tot_days,
            'outage_dur': season_df['Outage_Dur'].mean()
        }

        for combo in all_combos:
            combo_name = ''.join(combo)
            if filter_top_n_only and combo_name not in top_combo_names:
                continue

            days_sum = season_df[f'{combo_name}_Only_Day'].sum()
            outage_sum = season_df[f'{combo_name}_Only_Outage'].sum()

            row[f'{combo_name}_days'] = days_sum
            row[f'{combo_name}_n_outage'] = outage_sum
            row[f'{combo_name}_outage_prob'] = p_cond(outage_sum, days_sum)

        results.append(row)

    # All Seasons combined
    all_row = {
        'season': 'All Seasons',
        'total_days': len(df),
        'outage_dur': df['Outage_Dur'].mean()
    }

    for combo in all_combos:
        combo_name = ''.join(combo)
        if filter_top_n_only and combo_name not in top_combo_names:
            continue

        days_sum = combo_counts[combo_name]
        outage_sum = df[f'{combo_name}_Only_Outage'].sum()

        all_row[f'{combo_name}_days'] = days_sum
        all_row[f'{combo_name}_n_outage'] = outage_sum
        all_row[f'{combo_name}_outage_prob'] = p_cond(outage_sum, days_sum)

    results.append(all_row)

    # Print top N combinations
    if print_names: 
        print(f"\nTop {print_top_n} Most Frequent Exclusive Hazard Combinations:\n")
        for i, (combo_name, count) in enumerate(top_combos, 1):
            print(f"{i:>2}. {combo_name:<5} → {count} days")

    return pd.DataFrame(results).fillna(0).round(4)

In [ ]:
## statistical significance 

from math import sqrt
from scipy.stats import chi2_contingency

def weather_outage_chi2_cramersV(
    df,
    outage_col='Outage_Flag', 
    weather_vars = [
    'Hot_Day_Only', 'Windy_Day_Only', 'Rainy_Day_Only', 'Extreme_Lightning_Day_Only',
    'TL_Only', 'TP_Only', 'TW_Only', 'PL_Only', 'PW_Only', 'WL_Only',
    'TPL_Only', 'TPW_Only', 'TWL_Only', 'PWL_Only',
    'TPWL_Only'
    ], 
    percentile_val=90,
    duration_val=1,          
    overvolt_flag=False      # if True, use minutes instead of hours 
):
    # Derive grid metric name by stripping '_Flag' or '_flag' if present
    if outage_col.lower().endswith('_flag'):
        grid_metric = outage_col[:-5]
    else:
        grid_metric = outage_col

    outage_binary = (df[outage_col] > 0).astype(int)
    results = []

    # helper: assign significance stars
    def assign_significance(p, significant):
        if not significant:
            return ''
        if p < 0.01:
            return '***'
        elif p < 0.05:
            return '**'
        elif p <= 0.1:
            return '*'
        else:
            return ''

    for weather in weather_vars:
        if weather not in df.columns:
            print(f"Warning: '{weather}' not found in dataframe — skipping.")
            continue

        col_data = df[weather]
        if col_data.dtype == bool:
            col_data = col_data.astype(int)

        contingency = pd.crosstab(outage_binary, col_data)
        chi2, pval, dof, expected = chi2_contingency(contingency)

        n = contingency.values.sum()
        min_dim = min(contingency.shape) - 1
        cramers_v = sqrt(chi2 / (n * min_dim)) if min_dim > 0 else np.nan

        results.append({
            "Weather_Event": weather,
            "Test": "Chi-Square Test",
            "p-value": pval,
            "Cramers_V": cramers_v,
            "Contingency_Table": contingency
        })

    # Build summary dataframe
    results_df = pd.DataFrame([{
        "Weather_Event": r["Weather_Event"],
        "Significant": r["p-value"] < 0.05,
        "p-value": round(r["p-value"], 4),
        "Stars": assign_significance(r["p-value"], r["p-value"] < 0.05),
        "Cramers_V": r["Cramers_V"],
        "Grid_Metric": grid_metric
    } for r in results])

    results_df["Percentile"] = f"{percentile_val}th"

    if overvolt_flag:
        results_df["Duration (min)"] = duration_val
    else:
        results_df["Duration (hr)"] = duration_val

    return results_df, results

### Workflow - Outages 

In [ ]:
filtered_eas_sites_copy_r1 = merged_eas_sites.copy()

### 1+ hour outages 

#### 90th percentile 

In [ ]:
## Outage Duration & Percentile 
dur = 60  # duration in minutes
percentile = '90'

# Main loop
results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_df_TPLW_n_outage_data(
        row, 
        temp_all, 
        precip_all, 
        hourly_lightning_df, 
        hourly_wind_df, 
        pqr_hourly_all, 
        outage_threshold=dur
    )
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data_global = pd.concat(results, ignore_index=True)

# create daily summary of outage metrics & weather variables 
daily_agg_global = create_daily_summary(
    merged_hourly_data_global, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

# run global co-occurrence on all the TPLW combinations 
rez_co_occurrence_1hr_global_90th = calculate_cooccurrence_prob_all_combinations(daily_agg_global)
rez_co_occurrence_1hr_global_90th['Pct'] = f"{percentile}th"


Top 10 Most Frequent Exclusive Hazard Combinations:

 1. T     → 45164 days
 2. W     → 23638 days
 3. TW    → 13307 days
 4. P     → 6295 days
 5. PL    → 4751 days
 6. L     → 3463 days
 7. PW    → 2248 days
 8. TL    → 1626 days
 9. PWL   → 1063 days
10. WL    → 709 days


### Statistical Significance 

In [ ]:
# run exclusive version of daily summary 
daily_agg_global_XCL = create_daily_summary_exclusive(
    merged_hourly_data_global, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

# use output to calculate the statistical significance 
significance_results_90, __ = weather_outage_chi2_cramersV(
    df=daily_agg_global_XCL,
    outage_col='Outage_Flag', 
    duration_val=dur/60,
    percentile_val=90
)

#### 95th percentile 

In [ ]:
## Outage Duration & Percentile 
dur = 60  # duration in minutes
percentile = '95'

# Main loop
results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_df_TPLW_n_outage_data(
        row, 
        temp_all, 
        precip_all, 
        hourly_lightning_df, 
        hourly_wind_df, 
        pqr_hourly_all, 
        outage_threshold=dur
    )
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data_global = pd.concat(results, ignore_index=True)

daily_agg_global = create_daily_summary(
    merged_hourly_data_global, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

rez_co_occurrence_1hr_global_95th = calculate_cooccurrence_prob_all_combinations(daily_agg_global)
rez_co_occurrence_1hr_global_95th['Pct'] = f"{percentile}th"


Top 10 Most Frequent Exclusive Hazard Combinations:

 1. T     → 11460 days
 2. W     → 9015 days
 3. P     → 4621 days
 4. L     → 4449 days
 5. PL    → 2737 days
 6. TW    → 758 days
 7. PWL   → 313 days
 8. PW    → 285 days
 9. TL    → 230 days
10. WL    → 190 days


### Statistical Significance 

In [97]:
daily_agg_global_XCL = create_daily_summary_exclusive(
    merged_hourly_data_global, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

## enter outage_col, duration & percentile 
significance_results_95, __ = weather_outage_chi2_cramersV(
    df=daily_agg_global_XCL,
    outage_col='Outage_Flag', 
    duration_val=dur/60,
    percentile_val=95
)

#### 99th percentile 

In [ ]:
## Outage Duration & Percentile 
dur = 60  # duration in minutes
percentile = '99'

# Main loop
results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_df_TPLW_n_outage_data(
        row, 
        temp_all, 
        precip_all, 
        hourly_lightning_df, 
        hourly_wind_df, 
        pqr_hourly_all, 
        outage_threshold=dur
    )
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data_global = pd.concat(results, ignore_index=True)

daily_agg_global = create_daily_summary(
    merged_hourly_data_global, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile], 
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

rez_co_occurrence_1hr_global_99th = calculate_cooccurrence_prob_all_combinations(daily_agg_global)
rez_co_occurrence_1hr_global_99th['Pct'] = f"{percentile}th"


Top 10 Most Frequent Exclusive Hazard Combinations:

 1. L     → 1392 days
 2. P     → 1391 days
 3. T     → 565 days
 4. PL    → 382 days
 5. W     → 277 days
 6. WL    → 22 days
 7. PW    → 15 days
 8. TL    → 9 days
 9. PWL   → 6 days
10. TW    → 3 days


### Statistical Significance 

In [ ]:
daily_agg_global_XCL = create_daily_summary_exclusive(
    merged_hourly_data_global, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

significance_results_99, __ = weather_outage_chi2_cramersV(
    df=daily_agg_global_XCL,
    outage_col='Outage_Flag', 
    duration_val=dur/60,
    percentile_val=99
)

### Merge Statistical Significance Result Dfs - All Combos ***Wind*** 

In [ ]:
significance_results = [significance_results_90, significance_results_95, significance_results_99]
sig_rez_1hr_wind = pd.concat(significance_results, ignore_index=True)

### Merge Co_occurrence Dfs ***Wind*** 

In [ ]:
rez_1hr_list = [rez_co_occurrence_1hr_global_90th, rez_co_occurrence_1hr_global_95th, rez_co_occurrence_1hr_global_99th]
rez_1hr_wind = pd.concat(rez_1hr_list, ignore_index=True)

### 8+ hour outages 

#### 90th percentile 

In [ ]:
## Outage Duration & Percentile 
dur = 480  # duration in minutes
percentile = '90'

# Main loop
results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_df_TPLW_n_outage_data(
        row, 
        temp_all, 
        precip_all, 
        hourly_lightning_df, 
        hourly_wind_df, 
        pqr_hourly_all, 
        outage_threshold=dur
    )
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data_global = pd.concat(results, ignore_index=True)

daily_agg_global = create_daily_summary(
    merged_hourly_data_global, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

rez_co_occurrence_8hrs_global_90th = calculate_cooccurrence_prob_all_combinations(daily_agg_global)

rez_co_occurrence_8hrs_global_90th['Pct'] = f"{percentile}th"

### Statistical Significance 

In [ ]:
daily_agg_global_XCL = create_daily_summary_exclusive(
    merged_hourly_data_global, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

## enter outage_col, duration & percentile 
significance_results_90, __ = weather_outage_chi2_cramersV(
    df=daily_agg_global_XCL,
    outage_col='Outage_Flag', 
    duration_val=dur/60,
    percentile_val=90
)

#### 95th percentile 

In [ ]:
## Outage Duration & Percentile 
dur = 480  # duration in minutes
percentile = '95'

# Main loop
results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_df_TPLW_n_outage_data(
        row, 
        temp_all, 
        precip_all, 
        hourly_lightning_df, 
        hourly_wind_df, 
        pqr_hourly_all, 
        outage_threshold=dur
    )
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data_global = pd.concat(results, ignore_index=True)

daily_agg_global = create_daily_summary(
    merged_hourly_data_global, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

rez_co_occurrence_8hrs_global_95th = calculate_cooccurrence_prob_all_combinations(daily_agg_global)

rez_co_occurrence_8hrs_global_95th['Pct'] = f"{percentile}th"

### Statistical Significance 

In [ ]:
daily_agg_global_XCL = create_daily_summary_exclusive(
    merged_hourly_data_global, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

## enter outage_col, duration & percentile 
significance_results_95, __ = weather_outage_chi2_cramersV(
    df=daily_agg_global_XCL,
    outage_col='Outage_Flag', 
    duration_val=dur/60,
    percentile_val=95
)

#### 99th percentile 

In [ ]:
## Outage Duration & Percentile 
dur = 480  # duration in minutes
percentile = '99'

# Main loop
results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_df_TPLW_n_outage_data(
        row, 
        temp_all, 
        precip_all, 
        hourly_lightning_df, 
        hourly_wind_df, 
        pqr_hourly_all, 
        outage_threshold=dur
    )
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data_global = pd.concat(results, ignore_index=True)

daily_agg_global = create_daily_summary(
    merged_hourly_data_global, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

rez_co_occurrence_8hrs_global_99th = calculate_cooccurrence_prob_all_combinations(daily_agg_global)

rez_co_occurrence_8hrs_global_99th['Pct'] = f"{percentile}th"

### Statistical Significance 

In [ ]:
daily_agg_global_XCL = create_daily_summary_exclusive(
    merged_hourly_data_global, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

## enter outage_col, duration & percentile 
significance_results_99, __ = weather_outage_chi2_cramersV(
    df=daily_agg_global_XCL,
    outage_col='Outage_Flag', 
    duration_val=dur/60,
    percentile_val=99
)

### Merge Statistical Significance Result Dfs - All Combos ***Wind*** 

In [ ]:
significance_results = [significance_results_90, significance_results_95, significance_results_99]

sig_rez_8hr_wind = pd.concat(significance_results, ignore_index=True)

### Merge Co_occurrence Dfs ***Wind*** 

In [ ]:
rez_8hr_list = [rez_co_occurrence_8hrs_global_90th, rez_co_occurrence_8hrs_global_95th, rez_co_occurrence_8hrs_global_99th]

rez_8hr_wind = pd.concat(rez_8hr_list, ignore_index=True)

### Concatenating all outage co-occurrence dfs 

In [ ]:
outage_dfs = [rez_1hr_wind, rez_8hr_wind]

all_outage_rez_df_wind = pd.concat(outage_dfs, axis = 0, ignore_index=True)

# all_outage_rez_df_wind.to_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/all_outage_rez_with_wind_rev.csv')

### *** Concatenating all outage SIGNIFICANCE dfs *** 

In [ ]:
outage_sig_dfs = [sig_rez_1hr_wind, sig_rez_8hr_wind]

all_outage_SIGNIFICANCE_df_wind = pd.concat(outage_sig_dfs, axis = 0, ignore_index=True)

# all_outage_SIGNIFICANCE_df_wind.to_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/aall_outage_SIGNIFICANCE_df_6km_with_wind_rev.csv')

## Undervoltages Workflow 

### Functions 

In [ ]:
def flag_undervolt_hours(df):
    df = df.copy()
    df['Undervolt_Flag'] = df['total_undervolt_events'] > 0
    df['Undervolt_Dur'] = round(df['total_undervolt_duration'] / 60, 2)
    df = df[['time', 'site_id', 'Undervolt_Flag', 'Undervolt_Dur']]
    return df

In [ ]:
def prepare_hourly_df_TPLW_n_undervolt_data(ea_row, temp_df, precip_df, lightning_df, wind_df, pqr_df):
    ea = ea_row['ea_code9ch']
    site_list = ea_row['Intersecting_Sites']

    all_merged = []

    for site_id in site_list:
        site_id = int(site_id)

        # Filter temperature
        temp_filt = (
            temp_df[temp_df['ea_code9ch'] == ea]
            .reset_index(drop=True)
            .drop(columns=['geometry'], errors='ignore')
        )
        if 'Hot_Hour' in temp_filt.columns:
            temp_filt = temp_filt.drop(columns=['Hot_Hour'])

        temp_filt['time'] = temp_filt['time'].astype('datetime64[ns]')

        # Filter precipitation
        precip_filt = (
            precip_df[precip_df['ea_code9ch'] == ea]
            .reset_index(drop=True)
            .drop(columns=['geometry'], errors='ignore')
        )
        precip_filt['time'] = precip_filt['time'].astype('datetime64[ns]')
        precip_filt = precip_filt[['time', 'Precip']]

        # Merge temp and precip
        merged = temp_filt.merge(precip_filt, on='time', how='outer')

        # Filter lightning and outer join
        lightning_filt = (
            lightning_df[lightning_df['ea_code9ch'] == ea]
            .reset_index(drop=True)
            .drop(columns=['geometry'], errors='ignore')
            [['time', 'Lightning Events']]
        )
        merged = merged.merge(lightning_filt, on='time', how='outer')
        merged['Lightning Events'] = merged['Lightning Events'].fillna(0)

        ## Merge with Wind (inner merge)
        wind_filt = (
            wind_df[wind_df['ea_code9ch'] == ea]
            .reset_index(drop=True)
            .drop(columns=['geometry'], errors='ignore')
        )
        wind_filt = wind_filt[['time', 'Wind Gusts (m/s)']]  
        wind_filt['time'] = wind_filt['time'].astype('datetime64[ns]')

        merged = merged.merge(wind_filt, on='time', how='inner')

        # Filter undervolt data
        pqr_filt = (
            pqr_df[pqr_df['site_id'] == site_id]
            .reset_index(drop=True)[['time', 'site_id', 'total_undervolt_events', 'total_undervolt_duration']]
        )

        # Flag undervolts
        flagged_undervolts = flag_undervolt_hours(pqr_filt)

        # Inner join undervolts
        merged = merged.merge(flagged_undervolts, on='time', how='inner')

        # Add identifiers
        merged['ea_code9ch'] = ea
        merged['site_id'] = site_id

        # Reorder columns
        merged = merged[['time', 'Date', 'ea_code9ch', 'site_id', 'Temp', 'Precip',
                         'Lightning Events', 'Wind Gusts (m/s)',
                         'Undervolt_Flag', 'Undervolt_Dur']]

        all_merged.append(merged)

    return pd.concat(all_merged, ignore_index=True)

In [ ]:
## fixing double counting 

def create_daily_summary_unv(df, temp_thresh=32, hot_hours_thresh=5, precip_thresh=15, lightning_thresh = 42, wind_thresh = 5.93, windy_hours_thresh = 6):
    df = df.copy()
    df['Date'] = df['time'].dt.date
    df['Hot_hour'] = df['Temp'] > temp_thresh
    df['Windy_Hour'] = df['Wind Gusts (m/s)'] > wind_thresh

    daily_summary = (
        df.groupby(['Date', 'ea_code9ch'], as_index=False)
          .agg({
              'Undervolt_Flag': lambda x: (x > 0).any(),  # True if any site had undervolt
              'Undervolt_Dur': 'sum',
              'Precip': 'sum',
              'Lightning Events':'sum', 
              'Hot_hour': 'sum', 
              'Windy_Hour':'sum'
          })
    )

    ## Classify Days as High Temperature or High Precipitation 
    daily_summary['Hot_Day'] = daily_summary['Hot_hour'] >= hot_hours_thresh
    daily_summary['Windy_Day'] = daily_summary['Windy_Hour'] >= windy_hours_thresh

    daily_summary['Rainy_Day'] = daily_summary['Precip'] >= precip_thresh
    daily_summary['Extreme_Lightning_Day'] = daily_summary['Lightning Events'] >= lightning_thresh

    # Add Season Column
    daily_summary['Month'] = pd.to_datetime(daily_summary['Date']).dt.month
    daily_summary['Season'] = daily_summary['Month'].apply(assign_season)

    return daily_summary

In [ ]:
def create_daily_summary_exclusive_unv(
    df,
    temp_thresh,
    hot_hours_thresh,
    precip_thresh,
    lightning_thresh,
    wind_thresh,
    windy_hours_thresh
):
    df = df.copy()
    df['Date'] = df['time'].dt.date

    # Hourly flags
    df['Hot_hour'] = df['Temp'] > temp_thresh
    df['Windy_Hour'] = df['Wind Gusts (m/s)'] > wind_thresh

    # Aggregate daily
    daily_summary = (
        df.groupby(['Date', 'ea_code9ch'], as_index=False)
          .agg({
              'Undervolt_Flag': lambda x: (x > 0).any(),  # True if any site had undervolt
              'Undervolt_Dur': 'mean',
              'Precip': 'sum',
              'Lightning Events': 'sum',
              'Hot_hour': 'sum',
              'Windy_Hour': 'sum'
          })
    )

    # Classify weather event days
    daily_summary['Hot_Day'] = daily_summary['Hot_hour'] >= hot_hours_thresh
    daily_summary['Windy_Day'] = daily_summary['Windy_Hour'] >= windy_hours_thresh
    daily_summary['Rainy_Day'] = daily_summary['Precip'] >= precip_thresh
    daily_summary['Extreme_Lightning_Day'] = daily_summary['Lightning Events'] >= lightning_thresh

    # Exclusive single events
    daily_summary['Hot_Day_Only'] = (
        daily_summary['Hot_Day'] &
        ~daily_summary['Rainy_Day'] &
        ~daily_summary['Extreme_Lightning_Day'] &
        ~daily_summary['Windy_Day']
    )
    daily_summary['Rainy_Day_Only'] = (
        daily_summary['Rainy_Day'] &
        ~daily_summary['Hot_Day'] &
        ~daily_summary['Extreme_Lightning_Day'] &
        ~daily_summary['Windy_Day']
    )
    daily_summary['Extreme_Lightning_Day_Only'] = (
        daily_summary['Extreme_Lightning_Day'] &
        ~daily_summary['Hot_Day'] &
        ~daily_summary['Rainy_Day'] &
        ~daily_summary['Windy_Day']
    )
    daily_summary['Windy_Day_Only'] = (
        daily_summary['Windy_Day'] &
        ~daily_summary['Hot_Day'] &
        ~daily_summary['Rainy_Day'] &
        ~daily_summary['Extreme_Lightning_Day']
    )

    # Aliases
    T = daily_summary['Hot_Day']
    P = daily_summary['Rainy_Day']   # ✅ use Rainy_Day for P
    L = daily_summary['Extreme_Lightning_Day']
    W = daily_summary['Windy_Day']

    # Two-way combinations
    daily_summary['TL_Only'] = (T & L) & ~P & ~W
    daily_summary['TP_Only'] = (T & P) & ~L & ~W
    daily_summary['TW_Only'] = (T & W) & ~P & ~L
    daily_summary['PL_Only'] = (P & L) & ~T & ~W
    daily_summary['PW_Only'] = (P & W) & ~T & ~L
    daily_summary['WL_Only'] = (W & L) & ~T & ~P

    # Three-way combinations
    daily_summary['TPL_Only'] = (T & P & L) & ~W
    daily_summary['TPW_Only'] = (T & P & W) & ~L
    daily_summary['TWL_Only'] = (T & W & L) & ~P
    daily_summary['PWL_Only'] = (P & W & L) & ~T

    # Four-way combination
    daily_summary['TPWL_Only'] = (T & P & W & L)

    # Drop original event flags
    drop_cols = ['Hot_Day', 'Rainy_Day', 'Extreme_Lightning_Day', 'Windy_Day']
    daily_summary = daily_summary.drop(columns=[col for col in drop_cols if col in daily_summary.columns])

    return daily_summary

In [ ]:
def calculate_cooccurrence_prob_all_combinations_unv(
    df, 
    undervolt_dur=20,          # <-- passed in as a number
    print_top_n=10, 
    filter_top_n_only=True, 
    print_names=True
):
    df = df.copy()

    # Normalize season labels (merge Major/Minor rainy)
    df['Season'] = df['Season'].replace({
        'Major_Rainy_Season': 'Rainy_Season',
        'Minor_Rainy_Season': 'Rainy_Season'
    })

    # Binary undervolt flag
    df['Undervolt_Day'] = df['Undervolt_Flag'] > 0

    # Define hazards and their columns (added Wind 'W')
    hazard_cols = {
        'T': 'Hot_Day',
        'P': 'Rainy_Day',
        'W': 'Windy_Day', 
        'L': 'Extreme_Lightning_Day'
    }
    hazard_keys = list(hazard_cols.keys())

    # Generate all non-empty combinations of hazards
    all_combos = []
    for r in range(1, len(hazard_keys) + 1):
        all_combos.extend(combinations(hazard_keys, r))

    # Track combo counts
    combo_counts = {}
    for combo in all_combos:
        combo_name = ''.join(combo)
        in_combo = np.logical_and.reduce([df[hazard_cols[h]] for h in combo])
        not_in_combo = np.logical_not(
            np.logical_or.reduce([df[hazard_cols[h]] for h in hazard_keys if h not in combo])
        )
        df[f'{combo_name}_Only_Day'] = in_combo & not_in_combo
        df[f'{combo_name}_Only_Undervolt'] = df[f'{combo_name}_Only_Day'] & df['Undervolt_Day']
        combo_counts[combo_name] = df[f'{combo_name}_Only_Day'].sum()

    # Identify top-N most frequent combos
    top_combos = sorted(combo_counts.items(), key=lambda x: x[1], reverse=True)[:print_top_n]
    top_combo_names = {name for name, _ in top_combos}

    def p_cond(n, d): 
        return n / d if d else np.nan

    results = []

    # Per-season results
    for season in df['Season'].unique():
        season_df = df[df['Season'] == season]
        tot_days = len(season_df)

        row = {
            'season': season,
            'total_days': tot_days,
            'undervolt_dur': undervolt_dur   # <-- just use the passed-in number
        }

        for combo in all_combos:
            combo_name = ''.join(combo)
            if filter_top_n_only and combo_name not in top_combo_names:
                continue

            days_sum = season_df[f'{combo_name}_Only_Day'].sum()
            undervolt_sum = season_df[f'{combo_name}_Only_Undervolt'].sum()

            row[f'{combo_name}_days'] = days_sum
            row[f'{combo_name}_n_undervolt'] = undervolt_sum
            row[f'{combo_name}_undervolt_prob'] = p_cond(undervolt_sum, days_sum)

        results.append(row)

    # All Seasons combined
    all_row = {
        'season': 'All Seasons',
        'total_days': len(df),
        'undervolt_dur': undervolt_dur
    }

    for combo in all_combos:
        combo_name = ''.join(combo)
        if filter_top_n_only and combo_name not in top_combo_names:
            continue

        days_sum = combo_counts[combo_name]
        undervolt_sum = df[f'{combo_name}_Only_Undervolt'].sum()

        all_row[f'{combo_name}_days'] = days_sum
        all_row[f'{combo_name}_n_undervolt'] = undervolt_sum
        all_row[f'{combo_name}_undervolt_prob'] = p_cond(undervolt_sum, days_sum)

    results.append(all_row)

    # Print top-N
    if print_names: 
        print(f"\nTop {print_top_n} Most Frequent Exclusive Hazard Combinations:\n")
        for i, (combo_name, count) in enumerate(top_combos, 1):
            print(f"{i:>2}. {combo_name:<5} → {count} days")

    return pd.DataFrame(results).fillna(0).round(4)

### 60+min undervolts 

In [ ]:
# 22
volt_hourly_22 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/Geospatial_Voltage_Durations/Files/voltage_hourly_22_und_60.csv')
volt_hourly_22['time'] = pd.to_datetime(volt_hourly_22['time'])
volt_hourly_22['time'] = volt_hourly_22['time'].dt.tz_convert(None)


# 23
volt_hourly_23 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/Geospatial_Voltage_Durations/Files/voltage_hourly_23_und_60.csv')
volt_hourly_23['time'] = pd.to_datetime(volt_hourly_23['time'])
volt_hourly_23['time'] = volt_hourly_23['time'].dt.tz_convert(None)

# concatenate 
pqr_60_min_all = pd.concat([volt_hourly_22, volt_hourly_23], ignore_index=True)

## remove the incomplete sites 
pqr_60_min_all = pqr_60_min_all[ ~(pqr_60_min_all['site_id'].isin(incomplete_site_ids)) ]

### 1+ hour undervoltages  

#### 90th percentile 

In [ ]:
## Specify percentile 
dur = 60  # duration in minutes
percentile = '90'
voltage_df = pqr_60_min_all   ## 60 minutes / 1 hour 

results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_df_TPLW_n_undervolt_data(row, temp_all, precip_all, hourly_lightning_df, hourly_wind_df, voltage_df)
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data = pd.concat(results, ignore_index=True)

daily_agg = create_daily_summary_unv(
                            merged_hourly_data, 
                            temp_thresh=temp_thresh_dict[percentile], 
                            hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
                            precip_thresh=precip_thresh_dict[percentile], 
                            lightning_thresh=lightning_thresh_dict[percentile],  
                            wind_thresh=wind_thresh_dict[percentile], 
                            windy_hours_thresh=windy_hrs_thresh_dict[percentile]
                                       )

global_undervolt_60min_90th = calculate_cooccurrence_prob_all_combinations_unv(daily_agg, undervolt_dur = 60)

global_undervolt_60min_90th['Pct'] = f"{percentile}th"

### Statistical Significance

In [ ]:
daily_agg_global_XCL = create_daily_summary_exclusive_unv(
    merged_hourly_data, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

# enter outage_col, duration & percentile 
significance_results_90, __ = weather_outage_chi2_cramersV(
    df=daily_agg_global_XCL,
    outage_col='Undervolt_Flag', 
    duration_val=dur/60,
    percentile_val=90
)

#### 95th percentile 

In [ ]:
## Specify percentile 
dur = 60  # duration in minutes
percentile = '95'
voltage_df = pqr_60_min_all   ## 60 minutes / 1 hour 

results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_df_TPLW_n_undervolt_data(row, temp_all, precip_all, hourly_lightning_df, hourly_wind_df, voltage_df)
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data = pd.concat(results, ignore_index=True)

daily_agg = create_daily_summary_unv(
                            merged_hourly_data, 
                            temp_thresh=temp_thresh_dict[percentile], 
                            hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
                            precip_thresh=precip_thresh_dict[percentile], 
                            lightning_thresh=lightning_thresh_dict[percentile],  
                            wind_thresh=wind_thresh_dict[percentile], 
                            windy_hours_thresh=windy_hrs_thresh_dict[percentile]
                                       )

global_undervolt_60min_95th = calculate_cooccurrence_prob_all_combinations_unv(daily_agg, undervolt_dur = 60)

global_undervolt_60min_95th['Pct'] = f"{percentile}th"

### Statistical Significance

In [ ]:
daily_agg_global_XCL = create_daily_summary_exclusive_unv(
    merged_hourly_data, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

# enter outage_col, duration & percentile 
significance_results_95, __ = weather_outage_chi2_cramersV(
    df=daily_agg_global_XCL,
    outage_col='Undervolt_Flag', 
    duration_val=dur/60,
    percentile_val=95
)

#### 99th percentile 

In [ ]:
## Specify percentile 
dur = 60  # duration in minutes
percentile = '99'
voltage_df = pqr_60_min_all   ## 60 minutes / 1 hour 

results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_df_TPLW_n_undervolt_data(row, temp_all, precip_all, hourly_lightning_df, hourly_wind_df, voltage_df)
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data = pd.concat(results, ignore_index=True)

daily_agg = create_daily_summary_unv(
                            merged_hourly_data, 
                            temp_thresh=temp_thresh_dict[percentile], 
                            hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
                            precip_thresh=precip_thresh_dict[percentile], 
                            lightning_thresh=lightning_thresh_dict[percentile],  
                            wind_thresh=wind_thresh_dict[percentile], 
                            windy_hours_thresh=windy_hrs_thresh_dict[percentile]
                                       )

global_undervolt_60min_99th = calculate_cooccurrence_prob_all_combinations_unv(daily_agg, undervolt_dur = 60)

global_undervolt_60min_99th['Pct'] = f"{percentile}th"

### Statistical Significance

In [ ]:
daily_agg_global_XCL = create_daily_summary_exclusive_unv(
    merged_hourly_data, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

# enter outage_col, duration & percentile 
significance_results_99, __ = weather_outage_chi2_cramersV(
    df=daily_agg_global_XCL,
    outage_col='Undervolt_Flag', 
    duration_val=dur/60,
    percentile_val=99
)

### Merge Statistical Significance Result Dfs - All Combos ***Wind*** 

In [ ]:
significance_results = [significance_results_90, significance_results_95, significance_results_99]

sig_rez_1hr_unv_wind = pd.concat(significance_results, ignore_index=True)

### Merge Co_occurrence Dfs ***Wind*** 

In [ ]:
rez_unv_60min_list = [global_undervolt_60min_90th, global_undervolt_60min_95th, global_undervolt_60min_99th]

rez_unv_60min = pd.concat(rez_unv_60min_list, ignore_index=True)

### 240+min undervolts 

In [ ]:
# 22
volt_hourly_22 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/Geospatial_Voltage_Durations/Files/voltage_hourly_22_und_240.csv')
volt_hourly_22['time'] = pd.to_datetime(volt_hourly_22['time'])
volt_hourly_22['time'] = volt_hourly_22['time'].dt.tz_convert(None)


# 23
volt_hourly_23 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/Geospatial_Voltage_Durations/Files/voltage_hourly_23_und_240.csv')
volt_hourly_23['time'] = pd.to_datetime(volt_hourly_23['time'])
volt_hourly_23['time'] = volt_hourly_23['time'].dt.tz_convert(None)

pqr_240_min_all = pd.concat([volt_hourly_22, volt_hourly_23], ignore_index=True)

pqr_240_min_all = pqr_240_min_all[ ~(pqr_240_min_all['site_id'].isin(incomplete_site_ids)) ]

#### 90th percentile 

In [ ]:
## Specify percentile 
dur = 240  # duration in minutes
percentile = '90'
voltage_df = pqr_240_min_all   

results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_df_TPLW_n_undervolt_data(row, temp_all, precip_all, hourly_lightning_df, hourly_wind_df, voltage_df)
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data = pd.concat(results, ignore_index=True)

daily_agg = create_daily_summary_unv(
                            merged_hourly_data, 
                            temp_thresh=temp_thresh_dict[percentile], 
                            hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
                            precip_thresh=precip_thresh_dict[percentile], 
                            lightning_thresh=lightning_thresh_dict[percentile],  
                            wind_thresh=wind_thresh_dict[percentile], 
                            windy_hours_thresh=windy_hrs_thresh_dict[percentile]
                                       )

global_undervolt_240min_90th = calculate_cooccurrence_prob_all_combinations_unv(daily_agg, undervolt_dur = dur)

global_undervolt_240min_90th['Pct'] = f"{percentile}th"

### Statistical Significance

In [ ]:
daily_agg_global_XCL = create_daily_summary_exclusive_unv(
    merged_hourly_data, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

# enter outage_col, duration & percentile 
significance_results_90, __ = weather_outage_chi2_cramersV(
    df=daily_agg_global_XCL,
    outage_col='Undervolt_Flag', 
    duration_val=dur/60,
    percentile_val=90
)

#### 95th percentile 

In [ ]:
## Specify percentile 
dur = 240  # duration in minutes
percentile = '95'
voltage_df = pqr_240_min_all  

results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_df_TPLW_n_undervolt_data(row, temp_all, precip_all, hourly_lightning_df, hourly_wind_df, voltage_df)
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data = pd.concat(results, ignore_index=True)

daily_agg = create_daily_summary_unv(
                            merged_hourly_data, 
                            temp_thresh=temp_thresh_dict[percentile], 
                            hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
                            precip_thresh=precip_thresh_dict[percentile], 
                            lightning_thresh=lightning_thresh_dict[percentile],  
                            wind_thresh=wind_thresh_dict[percentile], 
                            windy_hours_thresh=windy_hrs_thresh_dict[percentile]
                                       )

global_undervolt_240min_95th = calculate_cooccurrence_prob_all_combinations_unv(daily_agg, undervolt_dur = dur)

global_undervolt_240min_95th['Pct'] = f"{percentile}th"

### Statistical Significance

In [ ]:
daily_agg_global_XCL = create_daily_summary_exclusive_unv(
    merged_hourly_data, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

# enter outage_col, duration & percentile 
significance_results_95, __ = weather_outage_chi2_cramersV(
    df=daily_agg_global_XCL,
    outage_col='Undervolt_Flag', 
    duration_val=dur/60,
    percentile_val=95
)

#### 99th percentile 

In [ ]:
## Specify percentile 
dur = 240  # duration in minutes
percentile = '99'
voltage_df = pqr_240_min_all  

results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_df_TPLW_n_undervolt_data(row, temp_all, precip_all, hourly_lightning_df, hourly_wind_df, voltage_df)
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data = pd.concat(results, ignore_index=True)

daily_agg = create_daily_summary_unv(
                            merged_hourly_data, 
                            temp_thresh=temp_thresh_dict[percentile], 
                            hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
                            precip_thresh=precip_thresh_dict[percentile], 
                            lightning_thresh=lightning_thresh_dict[percentile],  
                            wind_thresh=wind_thresh_dict[percentile], 
                            windy_hours_thresh=windy_hrs_thresh_dict[percentile]
                                       )

global_undervolt_240min_99th = calculate_cooccurrence_prob_all_combinations_unv(daily_agg, undervolt_dur = dur)

global_undervolt_240min_99th['Pct'] = f"{percentile}th"

### Statistical Significance

In [ ]:
daily_agg_global_XCL = create_daily_summary_exclusive_unv(
    merged_hourly_data, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

# enter outage_col, duration & percentile 
significance_results_99, __ = weather_outage_chi2_cramersV(
    df=daily_agg_global_XCL,
    outage_col='Undervolt_Flag', 
    duration_val=dur/60,
    percentile_val=99
)

### Merge Statistical Significance Result Dfs - All Combos ***Wind*** 

In [ ]:
significance_results = [significance_results_90, significance_results_95, significance_results_99]

sig_rez_4hr_unv_wind = pd.concat(significance_results, ignore_index=True)

### Merge Co_occurrence Dfs ***Wind*** 

In [ ]:
rez_unv_240min_list = [global_undervolt_240min_90th, global_undervolt_240min_95th, global_undervolt_240min_99th]

rez_unv_240min = pd.concat(rez_unv_240min_list, ignore_index=True)

### Concatenating ALL undervolt dfs 

In [ ]:
undervolt_dfs = [rez_unv_60min, rez_unv_240min]
all_undervolt_df = pd.concat(undervolt_dfs, axis = 0, ignore_index=True)

# all_undervolt_df.to_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/all_undervolt_df_with_wind_rev.csv')

### *** Concatenating all undervolt SIGNIFICANCE dfs *** 

In [ ]:
undervolt_sig_dfs = [sig_rez_1hr_unv_wind, sig_rez_4hr_unv_wind]
all_undervolt_SIGNIFICANCE_df_wind = pd.concat(undervolt_sig_dfs, axis = 0, ignore_index=True)

# all_undervolt_SIGNIFICANCE_df_wind.to_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/all_undervolt_SIGNIFICANCE_df_wind_6km_with_wind_rev.csv')